# Setup Detectors — Swing Points, Resistance & MA Entry

Python sandbox for fine-tuning detection logic before porting to the TS backend.

**Detectors included:**
1. ATR computation
2. Fractal pivot swing highs / swing lows (simple)
3. Significant swing points (prominence + departure)
4. Resistance / support level clustering
5. Moving-average pullback entry (EMA20 & SMA50)
6. Interactive chart with all overlays

In [235]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass, field
from typing import Literal

# If yfinance download fails with JSONDecodeError, upgrade it:
#   pip install --upgrade yfinance
# Minimum recommended version: 0.2.36+

## 1 — Fetch data

In [236]:
# Simple yfinance fetch
TICKER = "GDX"
PERIOD = "2y"      # 1mo, 3mo, 6mo, 1y, 2y, 5y, max
INTERVAL = "1d"     # 1d, 1wk, 1h

apple = yf.Ticker(TICKER)
raw = apple.history(period=PERIOD, interval=INTERVAL)

if raw.empty:
    raise ValueError("No data returned from yfinance. Try again later or change network/VPN.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[["Open", "High", "Low", "Close", "Volume"]].dropna().reset_index()
raw.rename(columns={"Date": "date", "Datetime": "date"}, inplace=True)

df = raw.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
})

# Keep a master copy for date-range re-runs (always refresh on new fetch)
df_all = df.copy()
DATA_TICKER = TICKER

print(df.tail())

                         date        open        high         low       close  \
497 2026-02-13 00:00:00-05:00  100.919998  104.139999   99.529999  103.940002   
498 2026-02-17 00:00:00-05:00   99.860001  101.190002   97.410004  100.250000   
499 2026-02-18 00:00:00-05:00  101.820000  103.690002  101.290001  102.559998   
500 2026-02-19 00:00:00-05:00  102.010002  104.300003  101.110001  104.239998   
501 2026-02-20 00:00:00-05:00  104.330002  106.389999  102.029999  106.260002   

       volume  
497  31928300  
498  26743000  
499  18924100  
500  18466500  
501  23431400  


## 2 — ATR & Average Bar Size

In [237]:
def true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr1 = df["high"] - df["low"]
    tr2 = (df["high"] - prev_close).abs()
    tr3 = (df["low"] - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def atr_series(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Simple rolling-mean ATR (matches the TS backend)."""
    tr = true_range(df)
    return tr.rolling(window=period, min_periods=period).mean()


def average_bar_size(df: pd.DataFrame, period: int = 20) -> pd.Series:
    """SMA of (High - Low) over `period` bars."""
    return (df["high"] - df["low"]).rolling(window=period, min_periods=1).mean()


df["atr14"] = atr_series(df, 14)
df["abs20"] = average_bar_size(df, 20)
df[["close", "atr14", "abs20"]].tail(5)

,close,atr14,abs20
497,103.940002,6.180714,4.740499
498,100.250000,6.254286,4.784499
499,102.559998,6.269286,4.798500
500,104.239998,5.865001,4.719500
501,106.260002,5.115001,4.698000


## 3 — Moving Averages

In [238]:
df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
df["sma50"] = df["close"].rolling(50).mean()
df["sma200"] = df["close"].rolling(200).mean()
df[["close", "ema20", "sma50", "sma200"]].tail(5)

,close,ema20,sma50,sma200
497,103.940002,99.996436,93.362388,69.623109
498,100.250000,100.020585,93.757878,69.881076
499,102.559998,100.262434,94.193015,70.159578
500,104.239998,100.641250,94.668900,70.447075
501,106.260002,101.176369,95.222316,70.736532


## 4 — Swing Point Detectors

### 4a — Fractal Pivots (simple one-sided lookahead)

Mirrors `detectFractalPivots` in the TS backend.

In [239]:
@dataclass
class SwingPoint:
    index: int
    price: float
    type: Literal["HIGH", "LOW"]
    atr: float = 0.0
    prominence: float = 0.0


def detect_fractal_pivots(
    df: pd.DataFrame,
    lookahead: int = 10,
) -> list[SwingPoint]:
    """
    Simple one-sided lookahead fractal pivots.
    Uses ABS (average bar size) as tolerance — same as the TS version.
    """
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = df["atr14"].values
    abs_vals = df["abs20"].values
    n = len(df)
    points: list[SwingPoint] = []

    for t in range(n - lookahead):
        abs_tol = abs_vals[t] if np.isfinite(abs_vals[t]) else 0
        a = atr_vals[t] if np.isfinite(atr_vals[t]) else abs_tol

        # Swing high
        is_high = all(highs[i] < highs[t] - abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_high:
            points.append(SwingPoint(index=t, price=highs[t], type="HIGH", atr=a))

        # Swing low
        is_low = all(lows[i] > lows[t] + abs_tol for i in range(t + 1, t + lookahead + 1))
        if is_low:
            points.append(SwingPoint(index=t, price=lows[t], type="LOW", atr=a))

    return points


fractal_pivots = detect_fractal_pivots(df, lookahead=10)
print(f"Fractal pivots found: {len(fractal_pivots)} "
      f"({sum(1 for p in fractal_pivots if p.type == 'HIGH')} highs, "
      f"{sum(1 for p in fractal_pivots if p.type == 'LOW')} lows)")

Fractal pivots found: 41 (8 highs, 33 lows)


### 4b — Significant Swing Points (ATR-gated with prominence + departure)

Mirrors `detectSignificantSwingPoints` in the TS backend.

In [240]:
def detect_significant_swings(
    df: pd.DataFrame,
    left: int = 3,
    right: int = 3,
    atr_period: int = 14,
    prom_atr: float = 1.5,
    depart_atr: float = 2.5,
    depart_lookahead: int = 10,
    min_swing_sep: int = 7,
) -> list[SwingPoint]:
    """
    4-condition significant swing detection:
      (A) Fractal: local max/min over [t-left .. t+right]
      (B) Prominence >= prom_atr * ATR
      (C) Departure: price moves >= depart_atr * ATR after the pivot
      (D) Spacing/dedup: keep most extreme within min_swing_sep bars
    """
    highs = df["high"].values
    lows = df["low"].values
    atr_vals = atr_series(df, atr_period).values
    n = len(df)
    candidates: list[SwingPoint] = []

    for t in range(left, n - right):
        a = atr_vals[t]
        if not np.isfinite(a) or a <= 0:
            continue

        window = slice(t - left, t + right + 1)

        # --- (A) Pivot high ---
        if highs[t] == highs[window].max() and all(
            highs[i] < highs[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_min_low = lows[window].min()
            prominence = highs[t] - local_min_low

            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                min_low_after = lows[t + 1 : dep_end].min() if t + 1 < dep_end else np.inf
                if min_low_after <= highs[t] - depart_atr * a:
                    candidates.append(SwingPoint(t, highs[t], "HIGH", a, prominence))

        # --- (A) Pivot low ---
        if lows[t] == lows[window].min() and all(
            lows[i] > lows[t] for i in range(t - left, t + right + 1) if i != t
        ):
            local_max_high = highs[window].max()
            prominence = local_max_high - lows[t]

            if prominence >= prom_atr * a:
                dep_end = min(n, t + depart_lookahead + 1)
                max_high_after = highs[t + 1 : dep_end].max() if t + 1 < dep_end else -np.inf
                if max_high_after >= lows[t] + depart_atr * a:
                    candidates.append(SwingPoint(t, lows[t], "LOW", a, prominence))

    # --- (D) Spacing / dedup ---
    candidates.sort(key=lambda p: p.index)
    result: list[SwingPoint] = []
    for p in candidates:
        if not result:
            result.append(p)
            continue
        last = result[-1]
        if p.type == last.type and p.index - last.index <= min_swing_sep:
            keep_new = p.price > last.price if p.type == "HIGH" else p.price < last.price
            if keep_new:
                result[-1] = p
        else:
            result.append(p)

    return result


# ── Tunable parameters ──
SIG_LEFT        = 3
SIG_RIGHT       = 3
SIG_PROM_ATR    = 1.5
SIG_DEPART_ATR  = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP     = 7

sig_swings = detect_significant_swings(
    df,
    left=SIG_LEFT,
    right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR,
    depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK,
    min_swing_sep=SIG_MIN_SEP,
)
print(f"Significant swings: {len(sig_swings)} "
      f"({sum(1 for p in sig_swings if p.type == 'HIGH')} highs, "
      f"{sum(1 for p in sig_swings if p.type == 'LOW')} lows)")

Significant swings: 47 (20 highs, 27 lows)


## 5 — Resistance / Support Level Clustering

Groups nearby swing-point prices into horizontal zones.  
A zone that gets tested multiple times is stronger resistance/support.

In [241]:
@dataclass
class Level:
    price: float
    type: Literal["RESISTANCE", "SUPPORT"]
    touches: int
    indices: list[int] = field(default_factory=list)


def cluster_levels(
    swings: list[SwingPoint],
    merge_pct: float = 0.015,
    min_touches: int = 2,
) -> list[Level]:
    """
    Cluster swing prices within `merge_pct` of each other.
    Returns horizontal S/R levels sorted by number of touches.
    """
    if not swings:
        return []

    sorted_swings = sorted(swings, key=lambda s: s.price)
    clusters: list[list[SwingPoint]] = [[sorted_swings[0]]]

    for sp in sorted_swings[1:]:
        cluster_avg = np.mean([s.price for s in clusters[-1]])
        if abs(sp.price - cluster_avg) / cluster_avg <= merge_pct:
            clusters[-1].append(sp)
        else:
            clusters.append([sp])

    levels: list[Level] = []
    for cluster in clusters:
        if len(cluster) < min_touches:
            continue
        avg_price = np.mean([s.price for s in cluster])
        high_count = sum(1 for s in cluster if s.type == "HIGH")
        low_count = sum(1 for s in cluster if s.type == "LOW")
        level_type = "RESISTANCE" if high_count >= low_count else "SUPPORT"
        levels.append(Level(
            price=round(avg_price, 2),
            type=level_type,
            touches=len(cluster),
            indices=[s.index for s in cluster],
        ))

    levels.sort(key=lambda lv: lv.touches, reverse=True)
    return levels


# ── Tunable parameters ──
MERGE_PCT   = 0.015   # 1.5% proximity to merge prices
MIN_TOUCHES = 2       # at least 2 touches to count as a level

all_swings = fractal_pivots + sig_swings
sr_levels = cluster_levels(all_swings, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)

print(f"S/R levels found: {len(sr_levels)}")
for lv in sr_levels:
    print(f"  {lv.type:>12s}  ${lv.price:>8.2f}  touches={lv.touches}")

S/R levels found: 22
       SUPPORT  $   35.11  touches=7
    RESISTANCE  $   37.33  touches=7
    RESISTANCE  $   38.36  touches=6
       SUPPORT  $   33.16  touches=5
       SUPPORT  $   35.96  touches=4
    RESISTANCE  $   83.85  touches=4
    RESISTANCE  $   41.07  touches=3
    RESISTANCE  $   42.50  touches=3
       SUPPORT  $   44.84  touches=3
       SUPPORT  $   46.64  touches=3
       SUPPORT  $   50.74  touches=3
    RESISTANCE  $   54.24  touches=3
       SUPPORT  $   91.63  touches=3
       SUPPORT  $   28.37  touches=2
    RESISTANCE  $   39.22  touches=2
    RESISTANCE  $   43.46  touches=2
       SUPPORT  $   49.67  touches=2
    RESISTANCE  $   52.82  touches=2
       SUPPORT  $   66.78  touches=2
       SUPPORT  $   73.55  touches=2
    RESISTANCE  $   78.95  touches=2
    RESISTANCE  $  113.50  touches=2


## 6 — Moving-Average Entry Detectors

### 6a — SMA50 Pullback (Stage 2)
Mirrors `PullbackDetector` — price near SMA50 with declining volume.

### 6b — EMA20 Pullback (Post-breakout)
Mirrors `Ema20PullbackDetector` — price near EMA20 after meaningful departure.

In [242]:
@dataclass
class EntrySignal:
    index: int
    date: object
    price: float
    type: str
    stop: float
    target: float
    rr: float
    metadata: dict = field(default_factory=dict)


def detect_sma50_pullback(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    vol_decline_bars: int = 3,
    stop_lookback: int = 5,
    target_rr: float = 5.0,
) -> list[EntrySignal]:
    """
    SMA50 pullback: price within `dist_atr_max` ATR of SMA50,
    volume declining over last `vol_decline_bars` bars.
    """
    signals: list[EntrySignal] = []
    closes = df["close"].values
    sma50 = df["sma50"].values
    atr = df["atr14"].values
    vols = df["volume"].values
    abs_vals = df["abs20"].values

    for t in range(max(stop_lookback, vol_decline_bars), len(df)):
        if not np.isfinite(sma50[t]) or not np.isfinite(atr[t]) or atr[t] <= 0:
            continue

        # Stage-2 proxy: close > SMA50 > SMA200 (if SMA200 available)
        sma200_val = df["sma200"].values[t] if "sma200" in df.columns else np.nan
        if np.isfinite(sma200_val) and sma50[t] <= sma200_val:
            continue

        dist = abs(closes[t] - sma50[t])
        if dist > dist_atr_max * atr[t]:
            continue

        # Volume declining
        recent_vols = vols[t - vol_decline_bars + 1 : t + 1]
        if not all(recent_vols[i] > recent_vols[i + 1] for i in range(len(recent_vols) - 1)):
            continue

        abs_val = abs_vals[t] if np.isfinite(abs_vals[t]) else atr[t]
        stop = float(df["low"].values[t - stop_lookback : t + 1].min() - abs_val)
        risk = closes[t] - stop
        if risk <= 0:
            continue

        signals.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="SMA50_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={"dist_atr": round(dist / atr[t], 2), "sma50": round(sma50[t], 2)},
        ))

    return signals


def detect_ema20_pullback(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Trend-following EMA20 bull pullback:
      - EMA20 > SMA50
      - meaningful departure >= departure_atr * ATR
      - touch-based entry: low <= ema20 <= high
      - close remains reasonably near EMA20
    """
    signals: list[EntrySignal] = []
    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        # EMA20 > SMA50 (ordered trend)
        if ema20[t] <= sma50[t]:
            continue

        # Meaningful departure: max close in lookback >= ema20 + departure_atr * ATR
        max_close = closes[t - departure_window : t + 1].max()
        if max_close < ema20[t] + departure_atr * atr[t]:
            continue

        # Touch-based pullback: candle must interact with EMA20
        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        if not touched_ema20:
            continue

        # Proximity guard
        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        stop = ema20[t] - 1.0 * atr[t]
        risk = closes[t] - stop
        if risk <= 0:
            continue

        signals.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="EMA20_TREND_FOLLOW_BULL_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_FOLLOWING_20EMA_BULL_PULLBACK",
                "dist_atr": round(dist / atr[t], 2),
                "ema20": round(ema20[t], 2),
                "departure_atr": round((max_close - ema20[t]) / atr[t], 2),
                "touched_ema20": touched_ema20,
            },
        ))

    return signals


def _price_efficiency(closes: np.ndarray) -> float:
    if len(closes) < 2:
        return 1.0
    net = abs(float(closes[-1] - closes[0]))
    path = float(np.abs(np.diff(closes)).sum())
    if path <= 1e-9:
        return 1.0
    return net / path


def _recent_volume_not_expanding(vols: np.ndarray, t: int, lookback: int = 5) -> bool:
    if t < 2 * lookback:
        return True
    recent = vols[t - lookback + 1 : t + 1]
    prior = vols[t - 2 * lookback + 1 : t - lookback + 1]
    if len(recent) < lookback or len(prior) < lookback:
        return True
    avg_recent = float(np.mean(recent))
    avg_prior = float(np.mean(prior))
    if avg_prior <= 0:
        return True
    return avg_recent <= 1.1 * avg_prior


def detect_pivot_pullback_long(
    df: pd.DataFrame,
    pivot_lookback: int = 40,
    breakout_lookback: int = 15,
    breakout_buffer_atr: float = 0.25,
    dist_atr_max: float = 1.0,
    stop_lookback: int = 5,
    target_rr: float = 4.0,
) -> list[EntrySignal]:
    """
    Pivot-pullback long:
      1) Stage-2 proxy (SMA50 > SMA200)
      2) A recent breakout above a prior pivot high
      3) Current bar pulls back near/retests that pivot
      4) No obvious volume expansion during pullback
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    atr = df["atr14"].values
    abs_vals = df["abs20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)

    start = pivot_lookback + breakout_lookback
    for t in range(start, len(df)):
        if not all(np.isfinite(v) for v in [atr[t], sma50[t]]):
            continue
        if atr[t] <= 0:
            continue

        if np.isfinite(sma200[t]) and sma50[t] <= sma200[t]:
            continue

        p0 = t - pivot_lookback - breakout_lookback
        p1 = t - breakout_lookback
        if p1 <= p0:
            continue

        pivot_price = float(np.max(highs[p0:p1]))

        breakout_closes = closes[p1 : t + 1]
        if len(breakout_closes) == 0:
            continue

        has_breakout = bool(np.any(breakout_closes >= pivot_price + breakout_buffer_atr * atr[t]))
        if not has_breakout:
            continue

        touched_pivot = lows[t] <= pivot_price <= highs[t]
        dist = abs(closes[t] - pivot_price)
        if not touched_pivot and dist > dist_atr_max * atr[t]:
            continue

        if closes[t] < sma50[t]:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        abs_val = abs_vals[t] if np.isfinite(abs_vals[t]) and abs_vals[t] > 0 else atr[t]
        stop = float(np.min(lows[t - stop_lookback + 1 : t + 1]) - abs_val)
        risk = closes[t] - stop
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="PIVOT_PULLBACK_LONG",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "PIVOT_PULLBACK_LONG",
                "direction": "LONG",
                "pivot": round(pivot_price, 2),
                "touched_pivot": touched_pivot,
                "dist_atr": round(dist / atr[t], 2),
                "breakout_buffer_atr": breakout_buffer_atr,
            },
        ))

    return out


def detect_trend_long_ema20_pullback_strict(
    df: pd.DataFrame,
    sig_swings: list[SwingPoint] | None = None,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Strict trend long EMA20 pullback:
      - Ordered trend: EMA20 > SMA50 and (if available) SMA50 > SMA200
      - Meaningful departure before pullback
      - Touch EMA20 + hold above EMA20 on close
      - Rejection-style candle and no expansion volume on pullback
      - Optional structure gate: recent significant swing high exists
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        if ema20[t] <= sma50[t]:
            continue
        if np.isfinite(sma200[t]) and sma50[t] <= sma200[t]:
            continue

        max_close = float(np.max(closes[t - departure_window : t + 1]))
        if max_close < ema20[t] + departure_atr * atr[t]:
            continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        if not touched_ema20:
            continue

        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        if closes[t] < ema20[t]:
            continue

        candle_range = max(highs[t] - lows[t], 1e-9)
        in_upper_half = closes[t] >= lows[t] + 0.5 * candle_range
        if not in_upper_half:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        if sig_swings is not None:
            recent_high = any(
                (s.type == "HIGH") and (t - departure_window <= s.index < t)
                for s in sig_swings
            )
            if not recent_high:
                continue

        stop = float(min(ema20[t] - atr[t], np.min(lows[max(0, t - 2) : t + 1]) - 0.2 * atr[t]))
        risk = closes[t] - stop
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="TREND_LONG_20EMA_PULLBACK",
            stop=round(stop, 2),
            target=round(closes[t] + risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_LONG_20EMA_PULLBACK_STRICT",
                "direction": "LONG",
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "departure_atr": round((max_close - ema20[t]) / atr[t], 2),
                "touched_ema20": touched_ema20,
                "in_upper_half": in_upper_half,
            },
        ))

    return out


def detect_base_failure_rally_short(
    df: pd.DataFrame,
    dist_atr_max: float = 1.0,
    failure_lookback: int = 20,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Base-failure rally short (first rally-back style):
      - Prior failure: close broke below SMA50 recently
      - Weak rally into SMA50 or EMA20 resistance
      - Prefer EMA20 < SMA50 structure, close remains below resistance MA
      - Weakness filter: low price efficiency and/or non-expanding volume
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    vols = df["volume"].values
    atr = df["atr14"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values

    for t in range(max(12, failure_lookback), len(df)):
        if not all(np.isfinite(v) for v in [atr[t], ema20[t], sma50[t]]):
            continue
        if atr[t] <= 0:
            continue

        look = slice(t - failure_lookback, t)
        broke_below_50 = bool(np.any(closes[look] < (sma50[look] - 0.25 * atr[look])))
        if not broke_below_50:
            continue

        if ema20[t] >= sma50[t]:
            continue

        recent_eff = _price_efficiency(closes[t - 10 : t + 1])
        weak_eff = recent_eff < 0.35
        weak_vol = _recent_volume_not_expanding(vols, t, lookback=5)
        if not (weak_eff or weak_vol):
            continue

        dist_sma50 = abs(highs[t] - sma50[t])
        dist_ema20 = abs(highs[t] - ema20[t])

        variant = None
        pivot = None
        if dist_sma50 <= dist_atr_max * atr[t] and closes[t] < sma50[t]:
            variant = "SMA50"
            pivot = float(sma50[t])
        elif dist_ema20 <= dist_atr_max * atr[t] and closes[t] < ema20[t]:
            ema20_slope_neg = ema20[t] < ema20[max(0, t - 5)]
            if not ema20_slope_neg:
                continue
            variant = "EMA20"
            pivot = float(ema20[t])

        if variant is None or pivot is None:
            continue

        stop = pivot + 0.5 * atr[t]
        risk = stop - closes[t]
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="BASE_FAILURE_SHORT",
            stop=round(stop, 2),
            target=round(closes[t] - risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "BASE_FAILURE_RALLY_SHORT",
                "direction": "SHORT",
                "ma_resistance": variant,
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "price_efficiency_10": round(recent_eff, 3),
                "weak_volume": weak_vol,
            },
        ))

    return out


def detect_trend_short_ema20_rally(
    df: pd.DataFrame,
    sig_swings: list[SwingPoint] | None = None,
    dist_atr_max: float = 1.0,
    departure_atr: float = 2.0,
    departure_window: int = 30,
    target_rr: float = 3.0,
) -> list[EntrySignal]:
    """
    Trend short EMA20 rally (mirror of long strict):
      - Ordered downtrend: EMA20 < SMA50
      - Meaningful downside departure from EMA20
      - Rally touches EMA20 then rejects below it
      - Distance and volume filters to reduce noise
    """
    out: list[EntrySignal] = []

    closes = df["close"].values
    highs = df["high"].values
    lows = df["low"].values
    vols = df["volume"].values
    ema20 = df["ema20"].values
    sma50 = df["sma50"].values
    sma200 = df["sma200"].values if "sma200" in df.columns else np.full(len(df), np.nan)
    atr = df["atr14"].values

    for t in range(departure_window, len(df)):
        if not all(np.isfinite(v) for v in [ema20[t], sma50[t], atr[t]]):
            continue
        if atr[t] <= 0:
            continue

        if ema20[t] >= sma50[t]:
            continue
        if np.isfinite(sma200[t]) and sma50[t] >= sma200[t]:
            continue

        min_close = float(np.min(closes[t - departure_window : t + 1]))
        if min_close > ema20[t] - departure_atr * atr[t]:
            continue

        touched_ema20 = lows[t] <= ema20[t] <= highs[t]
        if not touched_ema20:
            continue

        dist = abs(closes[t] - ema20[t])
        if dist > dist_atr_max * atr[t]:
            continue

        if closes[t] > ema20[t]:
            continue

        candle_range = max(highs[t] - lows[t], 1e-9)
        in_lower_half = closes[t] <= lows[t] + 0.5 * candle_range
        if not in_lower_half:
            continue

        if not _recent_volume_not_expanding(vols, t, lookback=3):
            continue

        if sig_swings is not None:
            recent_low = any(
                (s.type == "LOW") and (t - departure_window <= s.index < t)
                for s in sig_swings
            )
            if not recent_low:
                continue

        stop = float(max(ema20[t] + atr[t], np.max(highs[max(0, t - 2) : t + 1]) + 0.2 * atr[t]))
        risk = stop - closes[t]
        if risk <= 0:
            continue

        out.append(EntrySignal(
            index=t,
            date=df["date"].iloc[t],
            price=round(closes[t], 2),
            type="TREND_SHORT_20EMA_RALLY",
            stop=round(stop, 2),
            target=round(closes[t] - risk * target_rr, 2),
            rr=target_rr,
            metadata={
                "setup_class": "TREND_SHORT_20EMA_RALLY",
                "direction": "SHORT",
                "ema20": round(ema20[t], 2),
                "sma50": round(sma50[t], 2),
                "dist_atr": round(dist / atr[t], 2),
                "departure_atr": round((ema20[t] - min_close) / atr[t], 2),
                "touched_ema20": touched_ema20,
                "in_lower_half": in_lower_half,
            },
        ))

    return out


# ── Tunable parameters ──
SMA50_DIST_ATR   = 1.0   # max distance to SMA50 in ATR multiples
SMA50_VOL_BARS   = 3     # volume must decline over this many bars
SMA50_STOP_LOOK  = 5     # bars to look back for stop loss
SMA50_TARGET_RR  = 5.0   # risk-reward target

EMA20_DIST_ATR   = 1.0
EMA20_DEPART_ATR = 2.0   # minimum departure from EMA in ATR multiples
EMA20_DEPART_WIN = 30    # bars to look back for departure
EMA20_TARGET_RR  = 3.0

sma50_signals = detect_sma50_pullback(
    df,
    dist_atr_max=SMA50_DIST_ATR,
    vol_decline_bars=SMA50_VOL_BARS,
    stop_lookback=SMA50_STOP_LOOK,
    target_rr=SMA50_TARGET_RR,
)

ema20_signals = detect_ema20_pullback(
    df,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=EMA20_DEPART_ATR,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)

print(f"SMA50 pullback signals: {len(sma50_signals)}")
print(f"EMA20 pullback signals: {len(ema20_signals)}")

SMA50 pullback signals: 21
EMA20 pullback signals: 30


## 7 — Interactive Chart

In [243]:
# def build_chart(
#     df: pd.DataFrame,
#     fractal: list[SwingPoint],
#     sig: list[SwingPoint],
#     levels: list[Level],
#     sma50_sigs: list[EntrySignal],
#     ema20_sigs: list[EntrySignal],
#     title_suffix: str = "",
# ) -> go.Figure:
#     fig = make_subplots(
#         rows=2, cols=1, shared_xaxes=True,
#         row_heights=[0.75, 0.25], vertical_spacing=0.03,
#     )

#     # Candlestick
#     fig.add_trace(go.Candlestick(
#         x=df["date"], open=df["open"], high=df["high"],
#         low=df["low"], close=df["close"], name="Price",
#     ), row=1, col=1)

#     # Moving averages
#     for col, color, name in [
#         ("ema20", "#f59e0b", "EMA 20"),
#         ("sma50", "#3b82f6", "SMA 50"),
#         ("sma200", "#ef4444", "SMA 200"),
#     ]:
#         if col in df.columns:
#             fig.add_trace(go.Scatter(
#                 x=df["date"], y=df[col], name=name,
#                 line=dict(color=color, width=1.2),
#             ), row=1, col=1)

#     # Fractal pivots (already pre-filtered to active points)
#     fh = [p for p in fractal if p.type == "HIGH"]
#     fl = [p for p in fractal if p.type == "LOW"]
#     if fh:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in fh],
#             y=[p.price for p in fh],
#             mode="markers", name="Fractal High",
#             marker=dict(symbol="triangle-down", size=7, color="#f43f5e"),
#         ), row=1, col=1)
#     if fl:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in fl],
#             y=[p.price for p in fl],
#             mode="markers", name="Fractal Low",
#             marker=dict(symbol="triangle-up", size=7, color="#22c55e"),
#         ), row=1, col=1)

#     # Significant swings (already pre-filtered to active points)
#     sh = [p for p in sig if p.type == "HIGH"]
#     sl = [p for p in sig if p.type == "LOW"]
#     if sh:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in sh],
#             y=[p.price for p in sh],
#             mode="markers", name="Sig Swing High",
#             marker=dict(symbol="star", size=12, color="#dc2626", line=dict(width=1, color="white")),
#         ), row=1, col=1)
#     if sl:
#         fig.add_trace(go.Scatter(
#             x=[df["date"].iloc[p.index] for p in sl],
#             y=[p.price for p in sl],
#             mode="markers", name="Sig Swing Low",
#             marker=dict(symbol="star", size=12, color="#16a34a", line=dict(width=1, color="white")),
#         ), row=1, col=1)

#     # S/R levels as horizontal lines
#     for lv in levels:
#         color = "rgba(239,68,68,0.4)" if lv.type == "RESISTANCE" else "rgba(34,197,94,0.4)"
#         fig.add_hline(
#             y=lv.price, line_dash="dash", line_color=color, line_width=1,
#             annotation_text=f"{lv.type[0]} ${lv.price:.2f} (x{lv.touches})",
#             annotation_position="right", row=1, col=1,
#         )

#     # Entry signals (expected: current-bar valid only)
#     for sigs, name, color, symbol in [
#         (sma50_sigs, "SMA50 Setup (Current)", "#3b82f6", "diamond"),
#         (ema20_sigs, "EMA20 Setup (Current)", "#f59e0b", "diamond"),
#     ]:
#         if sigs:
#             fig.add_trace(go.Scatter(
#                 x=[df["date"].iloc[s.index] for s in sigs],
#                 y=[s.price for s in sigs],
#                 mode="markers", name=name,
#                 marker=dict(symbol=symbol, size=14, color=color,
#                             line=dict(width=2, color="white")),
#                 text=[f"Stop: {s.stop}  Target: {s.target}  RR: {s.rr}" for s in sigs],
#             ), row=1, col=1)

#     # Volume
#     colors = ["#22c55e" if df["close"].iloc[i] >= df["open"].iloc[i] else "#ef4444"
#               for i in range(len(df))]
#     fig.add_trace(go.Bar(
#         x=df["date"], y=df["volume"], name="Volume",
#         marker_color=colors, opacity=0.5,
#     ), row=2, col=1)

#     fig.update_layout(
#         title=f"{TICKER} — Setup Detectors{title_suffix}",
#         template="plotly_dark",
#         height=820,
#         xaxis_rangeslider_visible=False,
#         legend=dict(orientation="h", y=1.02, x=0),
#     )
#     fig.update_yaxes(title_text="Price", row=1, col=1)
#     fig.update_yaxes(title_text="Volume", row=2, col=1)
#     return fig


# chart = build_chart(df, fractal_pivots, sig_swings, sr_levels, [], [], title_suffix=" (All Data)")
# chart.show(renderer="notebook_connected")

## 8 — Signal Summary Table

In [244]:
all_signals = sma50_signals + ema20_signals
if all_signals:
    sig_df = pd.DataFrame([
        {"date": s.date, "type": s.type, "price": s.price,
         "stop": s.stop, "target": s.target, "rr": s.rr, **s.metadata}
        for s in all_signals
    ]).sort_values("date", ascending=False)
    display(sig_df.head(20))
else:
    print("No entry signals detected with current parameters.")

,date,type,price,stop,target,rr,dist_atr,sma50,setup_class,ema20,departure_atr,touched_ema20
50,2026-02-12 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,98.28,93.42,112.85,3.0,0.21,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,99.58,2.04,True
49,2026-02-09 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,102.89,92.59,133.79,3.0,0.70,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,98.64,2.23,True
48,2026-02-04 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,98.70,93.30,114.89,3.0,0.04,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,98.90,2.37,True
47,2026-02-03 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,98.22,93.55,112.24,3.0,0.13,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,98.92,2.47,True
46,2026-01-02 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,85.73,82.20,96.32,3.0,0.18,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,85.20,2.03,True
45,2025-12-29 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,85.85,81.75,98.14,3.0,0.31,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,84.89,2.04,True
20,2025-11-25 00:00:00-05:00,SMA50_PULLBACK,77.24,69.46,116.15,5.0,0.96,74.42,NaN,NaN,NaN,NaN
44,2025-11-20 00:00:00-05:00,EMA20_TREND_FOLLOW_BULL_PULLBACK,72.44,71.20,76.16,3.0,0.57,NaN,TREND_FOLLOWING_20EMA_BULL_PULLBACK,74.07,3.40,True
19,2025-11-19 00:00:00-05:00,SMA50_PULLBACK,75.84,70.43,102.88,5.0,0.71,73.96,NaN,NaN,NaN,NaN
18,2025-11-18 00:00:00-05:00,SMA50_PULLBACK,75.08,70.41,98.45,5.0,0.48,73.80,NaN,NaN,NaN,NaN


## 9 — Quick Back-test: Forward Returns After Signals

In [245]:
def forward_returns(df: pd.DataFrame, signals: list[EntrySignal], horizons: list[int] = [5, 10, 20]) -> pd.DataFrame:
    """Compute forward returns after each signal to gauge effectiveness."""
    rows = []
    closes = df["close"].values
    for s in signals:
        row = {"date": s.date, "type": s.type, "entry": s.price}
        for h in horizons:
            if s.index + h < len(closes):
                ret = (closes[s.index + h] - s.price) / s.price * 100
                row[f"{h}d_ret_%"] = round(ret, 2)
            else:
                row[f"{h}d_ret_%"] = None

        # Hit target?
        hit_target = False
        hit_stop = False
        for j in range(s.index + 1, min(s.index + max(horizons) + 1, len(df))):
            if df["high"].iloc[j] >= s.target:
                hit_target = True
                break
            if df["low"].iloc[j] <= s.stop:
                hit_stop = True
                break

        row["outcome"] = "TARGET" if hit_target else "STOPPED" if hit_stop else "OPEN"
        rows.append(row)

    return pd.DataFrame(rows)


if all_signals:
    fwd = forward_returns(df, all_signals)
    print(f"\nSignal outcomes (within {20}d window):")
    print(fwd["outcome"].value_counts().to_string())
    print()
    display(fwd)
else:
    print("No signals to back-test.")


Signal outcomes (within 20d window):
outcome
OPEN       24
STOPPED    20
TARGET      7



,date,type,entry,5d_ret_%,10d_ret_%,20d_ret_%,outcome
0,2024-06-11 00:00:00-04:00,SMA50_PULLBACK,33.12,-0.15,0.20,12.85,OPEN
1,2024-06-25 00:00:00-04:00,SMA50_PULLBACK,33.21,0.52,9.47,10.01,OPEN
2,2024-08-09 00:00:00-04:00,SMA50_PULLBACK,35.17,7.39,9.76,2.56,OPEN
3,2024-09-10 00:00:00-04:00,SMA50_PULLBACK,36.52,6.11,11.27,4.76,OPEN
4,2024-09-11 00:00:00-04:00,SMA50_PULLBACK,36.69,4.47,10.48,3.74,OPEN
5,2024-10-07 00:00:00-04:00,SMA50_PULLBACK,38.30,2.89,10.73,2.23,OPEN
6,2024-11-04 00:00:00-05:00,SMA50_PULLBACK,39.15,-7.94,-7.21,-5.51,STOPPED
7,2024-11-08 00:00:00-05:00,SMA50_PULLBACK,38.29,-9.00,-1.90,-2.85,STOPPED
8,2025-04-08 00:00:00-04:00,SMA50_PULLBACK,41.20,21.76,17.42,20.05,OPEN
9,2025-05-16 00:00:00-04:00,SMA50_PULLBACK,46.00,9.06,16.05,15.57,OPEN


## 10 — Parameter Tuning Playground

Re-run any detector with different parameters and see the results instantly.

In [246]:
# New runner: date range + active-only swings + current MA setup only

# Date range filter (set None for full range)
START_DATE = None        # e.g. "2025-01-01"
END_DATE = None          # e.g. "2026-02-15"

# Fractal pivot sensitivity
FRACTAL_LOOKAHEAD = 10

# Significant swing thresholds
SIG_LEFT = 3
SIG_RIGHT = 3
SIG_PROM_ATR = 1.5
SIG_DEPART_ATR = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP = 7

# S/R clustering
MERGE_PCT = 0.015
MIN_TOUCHES = 2

# MA setup thresholds
SMA50_DIST_ATR = 1.0
SMA50_VOL_BARS = 3
SMA50_STOP_LOOK = 5
SMA50_TARGET_RR = 5.0

EMA20_DIST_ATR = 1.0
EMA20_DEPART_ATR = 2.0
EMA20_DEPART_WIN = 30
EMA20_TARGET_RR = 3.0


def apply_date_range(df_src: pd.DataFrame, start_date=None, end_date=None) -> pd.DataFrame:
    out = df_src.copy()
    if start_date is not None:
        out = out[out["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        out = out[out["date"] <= pd.to_datetime(end_date)]
    return out.reset_index(drop=True)


def add_indicators(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out["atr14"] = atr_series(out, 14)
    out["abs20"] = average_bar_size(out, 20)
    out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    out["sma50"] = out["close"].rolling(50).mean()
    out["sma200"] = out["close"].rolling(200).mean()
    return out


def split_active_expired_swings(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    latest_atr: float,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """
    Split swings into active + expired_high + expired_low.

    Path-based expiration (not just current-price snapshot):
      - HIGH expires if any future HIGH breaches swing_high + atr_buffer*ATR_ref
      - LOW  expires if any future LOW breaches swing_low  - atr_buffer*ATR_ref

    ATR_ref is swing ATR when available; fallback to atr14 at swing bar; fallback to latest ATR.
    """
    active: list[SwingPoint] = []
    expired_high: list[SwingPoint] = []
    expired_low: list[SwingPoint] = []

    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values
    n = len(df_ctx)

    for s in swings:
        if s.index >= n - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = latest_atr if np.isfinite(latest_atr) and latest_atr > 0 else 0.0

        thr = atr_buffer * atr_ref

        if s.type == "HIGH":
            breach_level = s.price + thr
            breached = np.any(highs[s.index + 1 :] > breach_level)
            if breached:
                expired_high.append(s)
            else:
                active.append(s)
        else:
            breach_level = s.price - thr
            breached = np.any(lows[s.index + 1 :] < breach_level)
            if breached:
                expired_low.append(s)
            else:
                active.append(s)

    return active, expired_high, expired_low


# Use master data if available, otherwise current df
if "df_all" not in globals():
    df_all = df.copy()

df_run = apply_date_range(df_all, START_DATE, END_DATE)
if len(df_run) < 60:
    raise ValueError("Date range too small. Use at least ~60 bars.")

df_run = add_indicators(df_run)
latest_close = float(df_run["close"].iloc[-1])
latest_atr = float(df_run["atr14"].iloc[-1]) if np.isfinite(df_run["atr14"].iloc[-1]) else 0.0

# Detect swings
fractal_pivots = detect_fractal_pivots(df_run, lookahead=FRACTAL_LOOKAHEAD)
sig_swings = detect_significant_swings(
    df_run,
    left=SIG_LEFT,
    right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR,
    depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK,
    min_swing_sep=SIG_MIN_SEP,
)

# Split swings into active + expired (ATR invalidated)
fractal_active, fractal_expired_high, fractal_expired_low = split_active_expired_swings(
    fractal_pivots, df_run, latest_atr, atr_buffer=1.0
)
sig_active, sig_expired_high, sig_expired_low = split_active_expired_swings(
    sig_swings, df_run, latest_atr, atr_buffer=1.0
)
expired_high_all = fractal_expired_high + sig_expired_high
expired_low_all = fractal_expired_low + sig_expired_low

# S/R from active swings
sr_levels = cluster_levels(fractal_active + sig_active, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)

# Detect MA setups, keep current valid, and label past ones as expired
sma50_all = detect_sma50_pullback(df_run, SMA50_DIST_ATR, SMA50_VOL_BARS, SMA50_STOP_LOOK, SMA50_TARGET_RR)
ema20_all = detect_ema20_pullback(df_run, EMA20_DIST_ATR, EMA20_DEPART_ATR, EMA20_DEPART_WIN, EMA20_TARGET_RR)

last_idx = len(df_run) - 1
sma50_current = [s for s in sma50_all if s.index == last_idx]
ema20_current = [s for s in ema20_all if s.index == last_idx]

# Expired MA entries = historical entries not active on current bar
sma50_expired = [s for s in sma50_all if s.index < last_idx]
ema20_expired = [s for s in ema20_all if s.index < last_idx]

range_text = f" [{df_run['date'].iloc[0]} -> {df_run['date'].iloc[-1]}]"
print(f"Bars used: {len(df_run)}{range_text}")
print(f"Latest close: {latest_close:.2f}")
print(f"Latest ATR14: {latest_atr:.2f}")
print(f"Active fractal swings: {len(fractal_active)}")
print(f"Active significant swings: {len(sig_active)}")
print(f"Expired swing highs (1 ATR): {len(expired_high_all)}")
print(f"Expired swing lows (1 ATR): {len(expired_low_all)}")
print(f"Active S/R levels: {len(sr_levels)}")
print(f"Current SMA50 setup valid: {len(sma50_current) > 0}")
print(f"Current EMA20 setup valid: {len(ema20_current) > 0}")
print(f"Expired SMA50 entries: {len(sma50_expired)}")
print(f"Expired EMA20 entries: {len(ema20_expired)}")

# Keep globals updated for follow-up cells
df = df_run
sma50_signals = sma50_current
ema20_signals = ema20_current


def build_ma_entry_chart(
    df_plot: pd.DataFrame,
    sma_sigs: list[EntrySignal],
    ema_sigs: list[EntrySignal],
    sma_expired: list[EntrySignal],
    ema_expired: list[EntrySignal],
) -> go.Figure:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.03)

    fig.add_trace(go.Candlestick(
        x=df_plot["date"], open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"], close=df_plot["close"], name="Price"
    ), row=1, col=1)

    fig.add_trace(go.Scatter(x=df_plot["date"], y=df_plot["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.6)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_plot["date"], y=df_plot["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.6)), row=1, col=1)

    # Expired MA setups (historical, not current)
    if sma_expired:
        fig.add_trace(go.Scatter(
            x=[df_plot["date"].iloc[s.index] for s in sma_expired],
            y=[s.price for s in sma_expired],
            mode="markers", name="SMA50 Expired",
            marker=dict(symbol="x", size=9, color="rgba(59,130,246,0.55)"),
        ), row=1, col=1)

    if ema_expired:
        fig.add_trace(go.Scatter(
            x=[df_plot["date"].iloc[s.index] for s in ema_expired],
            y=[s.price for s in ema_expired],
            mode="markers", name="EMA20 Expired",
            marker=dict(symbol="x", size=9, color="rgba(245,158,11,0.55)"),
        ), row=1, col=1)

    # Show current valid setups (if any)
    if sma_sigs:
        s = sma_sigs[0]
        x0 = df_plot["date"].iloc[s.index]
        fig.add_trace(go.Scatter(
            x=[x0], y=[s.price], mode="markers", name="SMA50 Current Setup",
            marker=dict(symbol="diamond", size=14, color="#3b82f6", line=dict(width=2, color="white")),
        ), row=1, col=1)
        fig.add_hline(y=s.stop, line_dash="dot", line_color="#3b82f6", annotation_text=f"SMA50 stop {s.stop:.2f}", row=1, col=1)
        fig.add_hline(y=s.target, line_dash="dot", line_color="#60a5fa", annotation_text=f"SMA50 target {s.target:.2f}", row=1, col=1)

    if ema_sigs:
        s = ema_sigs[0]
        x0 = df_plot["date"].iloc[s.index]
        fig.add_trace(go.Scatter(
            x=[x0], y=[s.price], mode="markers", name="EMA20 Current Setup",
            marker=dict(symbol="diamond", size=14, color="#f59e0b", line=dict(width=2, color="white")),
        ), row=1, col=1)
        fig.add_hline(y=s.stop, line_dash="dot", line_color="#f59e0b", annotation_text=f"EMA20 stop {s.stop:.2f}", row=1, col=1)
        fig.add_hline(y=s.target, line_dash="dot", line_color="#fbbf24", annotation_text=f"EMA20 target {s.target:.2f}", row=1, col=1)

    vol_colors = ["#22c55e" if df_plot["close"].iloc[i] >= df_plot["open"].iloc[i] else "#ef4444" for i in range(len(df_plot))]
    fig.add_trace(go.Bar(x=df_plot["date"], y=df_plot["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

    fig.update_layout(
        title=f"{TICKER} — MA Entry Chart{range_text}",
        template="plotly_dark",
        height=760,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    return fig


# Chart 1: Full setup context (active swings + expired swings + MA states)
# Old chart removed (use final classification chart cell)
# chart = build_chart(
#     df_run,
#     fractal_active,
#     sig_active,
#     sr_levels,
#     sma50_current,
#     ema20_current,
#     title_suffix=range_text,
# )

# Old chart blocks removed. Use the final MA Setup Classification chart cell only.
print("Old chart blocks removed. Run the final MA Setup Classification cell only.")

Bars used: 502 [2024-02-21 00:00:00-05:00 -> 2026-02-20 00:00:00-05:00]
Latest close: 106.26
Latest ATR14: 5.12
Active fractal swings: 27
Active significant swings: 19
Expired swing highs (1 ATR): 26
Expired swing lows (1 ATR): 16
Active S/R levels: 9
Current SMA50 setup valid: False
Current EMA20 setup valid: False
Expired SMA50 entries: 21
Expired EMA20 entries: 30
Old chart blocks removed. Run the final MA Setup Classification cell only.


In [247]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TUNE THESE — re-run this cell + the chart cell to compare ║
# ╚══════════════════════════════════════════════════════════════╝

# Fractal pivot sensitivity
FRACTAL_LOOKAHEAD = 10

# Significant swing thresholds
SIG_LEFT        = 3
SIG_RIGHT       = 3
SIG_PROM_ATR    = 1.5
SIG_DEPART_ATR  = 2.5
SIG_DEPART_LOOK = 10
SIG_MIN_SEP     = 7

# S/R clustering
MERGE_PCT   = 0.015
MIN_TOUCHES = 2

# MA entry detectors
SMA50_DIST_ATR   = 1.0
SMA50_VOL_BARS   = 3
SMA50_STOP_LOOK  = 5
SMA50_TARGET_RR  = 5.0

EMA20_DIST_ATR   = 1.0
EMA20_DEPART_ATR = 2.0
EMA20_DEPART_WIN = 30
EMA20_TARGET_RR  = 3.0

# ── Re-detect ──
fractal_pivots = detect_fractal_pivots(df, lookahead=FRACTAL_LOOKAHEAD)
sig_swings = detect_significant_swings(
    df, left=SIG_LEFT, right=SIG_RIGHT,
    prom_atr=SIG_PROM_ATR, depart_atr=SIG_DEPART_ATR,
    depart_lookahead=SIG_DEPART_LOOK, min_swing_sep=SIG_MIN_SEP,
)
sr_levels = cluster_levels(fractal_pivots + sig_swings, merge_pct=MERGE_PCT, min_touches=MIN_TOUCHES)
sma50_signals = detect_sma50_pullback(df, SMA50_DIST_ATR, SMA50_VOL_BARS, SMA50_STOP_LOOK, SMA50_TARGET_RR)
ema20_signals = detect_ema20_pullback(df, EMA20_DIST_ATR, EMA20_DEPART_ATR, EMA20_DEPART_WIN, EMA20_TARGET_RR)

print(f"Fractal pivots: {len(fractal_pivots)}")
print(f"Significant swings: {len(sig_swings)}")
print(f"S/R levels: {len(sr_levels)}")
print(f"SMA50 entries: {len(sma50_signals)}")
print(f"EMA20 entries: {len(ema20_signals)}")

print("Deprecated tuning chart removed. Use the final MA Setup Classification chart cell.")

Fractal pivots: 41
Significant swings: 47
S/R levels: 22
SMA50 entries: 21
EMA20 entries: 30
Deprecated tuning chart removed. Use the final MA Setup Classification chart cell.


In [248]:
# FINAL RUNNER v2: Daily Base (spec) + merged bases + all expired overlays
# Use THIS as the last chart cell.

START_DATE = None
END_DATE = None


def _slice_df(df_src: pd.DataFrame, start_date=None, end_date=None) -> pd.DataFrame:
    out = df_src.copy()
    if start_date is not None:
        out = out[out["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        out = out[out["date"] <= pd.to_datetime(end_date)]
    return out.reset_index(drop=True)


def _ensure_indicators(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "atr14" not in out.columns:
        out["atr14"] = atr_series(out, 14)
    if "abs20" not in out.columns:
        out["abs20"] = average_bar_size(out, 20)
    if "ema20" not in out.columns:
        out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    if "sma50" not in out.columns:
        out["sma50"] = out["close"].rolling(50).mean()
    if "sma200" not in out.columns:
        out["sma200"] = out["close"].rolling(200).mean()
    return out


def split_swings_for_chart(
    swings: list[SwingPoint],
    df_ctx: pd.DataFrame,
    atr_buffer: float = 1.0,
) -> tuple[list[SwingPoint], list[SwingPoint], list[SwingPoint]]:
    """Return (active, expired_high, expired_low) using path-based ATR invalidation."""
    active, expired_high, expired_low = [], [], []
    highs = df_ctx["high"].values
    lows = df_ctx["low"].values
    atr_arr = df_ctx["atr14"].values

    for s in swings:
        if s.index >= len(df_ctx) - 1:
            active.append(s)
            continue

        atr_ref = s.atr if np.isfinite(s.atr) and s.atr > 0 else atr_arr[s.index]
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            atr_ref = atr_arr[-1] if np.isfinite(atr_arr[-1]) and atr_arr[-1] > 0 else 0.0

        thr = atr_buffer * atr_ref
        if s.type == "HIGH":
            if np.any(highs[s.index + 1 :] > s.price + thr):
                expired_high.append(s)
            else:
                active.append(s)
        else:
            if np.any(lows[s.index + 1 :] < s.price - thr):
                expired_low.append(s)
            else:
                active.append(s)

    return active, expired_high, expired_low


def detect_daily_base_candidates(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    min_base_duration: int = 20,
    max_retrace: float = 0.50,
    contraction_tail: int = 5,
    min_window_bars: int = 5,
    min_window_gap: int = 3,
) -> list[dict]:
    """
    Stateful BASE rule from swing high:
      - Start at swing HIGH (peak)
      - Stay in BASE until first invalidation:
          1) retrace > max_retrace
          2) close < sma200
          3) high > (peak + 1*ATR)
      - Keep candidate only if duration >= min_base_duration

    This flags the full base span (not only the downswing leg).
    """
    highs = df["high"].values
    lows = df["low"].values
    closes = df["close"].values
    sma200 = df["sma200"].values
    atr = df["atr14"].values

    n = len(df)
    peak_swings = sorted([s for s in swings if s.type == "HIGH"], key=lambda s: s.index)
    out: list[dict] = []

    for pk in peak_swings:
        p = pk.index
        peak_price = float(pk.price)

        if p >= n - min_base_duration:
            continue

        atr_ref = atr[p] if np.isfinite(atr[p]) and atr[p] > 0 else np.nan
        if not np.isfinite(atr_ref) or atr_ref <= 0:
            # without ATR we cannot apply breakout = peak + 1*ATR reliably
            continue

        breakout_level = peak_price + atr_ref
        span_low = peak_price
        last_valid_idx = None
        invalid_reason = None

        for t in range(p, n):
            span_low = min(span_low, float(lows[t]))
            retrace = (peak_price - span_low) / peak_price if peak_price > 0 else 1.0

            broke_retrace = retrace > max_retrace
            below_200 = (not np.isfinite(sma200[t])) or (closes[t] < sma200[t])
            broke_out = highs[t] > breakout_level

            if broke_retrace:
                invalid_reason = "retrace_gt_50pct"
                break
            if below_200:
                invalid_reason = "close_below_sma200"
                break
            if broke_out:
                invalid_reason = "breakout_above_peak_plus_1atr"
                break

            last_valid_idx = t

        if last_valid_idx is None:
            continue

        duration = last_valid_idx - p + 1
        if duration < min_base_duration:
            continue

        base_high = float(np.max(highs[p : last_valid_idx + 1]))
        base_low = float(np.min(lows[p : last_valid_idx + 1]))
        retrace = (peak_price - base_low) / peak_price if peak_price > 0 else 1.0

        mid = max(p + 1, last_valid_idx - contraction_tail + 1)
        c_high = float(np.max(highs[mid : last_valid_idx + 1]))
        c_low = float(np.min(lows[mid : last_valid_idx + 1]))
        base_h = max(base_high - base_low, 1e-9)
        c_ratio = (c_high - c_low) / base_h

        out.append({
            "peak_idx": int(p),
            "peak_price": round(peak_price, 2),
            "start_idx": int(p),
            "mid_idx": int(mid),
            "end_idx": int(last_valid_idx),
            "base_high": round(base_high, 2),
            "base_low": round(base_low, 2),
            "contraction_high": round(c_high, 2),
            "contraction_low": round(c_low, 2),
            "contraction_ratio": round(c_ratio, 2),
            "retrace_pct": round(retrace, 3),
            "duration": int(duration),
            "breakout_level": round(float(breakout_level), 2),
            "invalid_reason": invalid_reason if invalid_reason is not None else "active_to_end_of_data",
        })

    return out


def merge_overlapping_bases(candidates: list[dict], df: pd.DataFrame, contraction_tail: int = 5) -> list[dict]:
    if not candidates:
        return []

    # one representative per peak (latest valid window)
    by_peak: dict[int, dict] = {}
    for c in candidates:
        pk = c["peak_idx"]
        cur = by_peak.get(pk)
        if cur is None or c["end_idx"] > cur["end_idx"]:
            by_peak[pk] = c

    reps = sorted(by_peak.values(), key=lambda x: x["start_idx"])
    merged: list[dict] = []

    highs = df["high"].values
    lows = df["low"].values

    for c in reps:
        if not merged:
            merged.append(c.copy())
            continue

        last = merged[-1]
        t0 = max(last["start_idx"], c["start_idx"])
        t1 = min(last["end_idx"], c["end_idx"])
        time_overlap = max(0, t1 - t0 + 1)
        min_len = max(1, min(last["end_idx"] - last["start_idx"] + 1, c["end_idx"] - c["start_idx"] + 1))
        t_overlap_ratio = time_overlap / min_len

        p0 = max(last["base_low"], c["base_low"])
        p1 = min(last["base_high"], c["base_high"])
        price_overlap = max(0.0, p1 - p0)
        min_h = max(1e-9, min(last["base_high"] - last["base_low"], c["base_high"] - c["base_low"]))
        p_overlap_ratio = price_overlap / min_h

        if t_overlap_ratio >= 0.60 and p_overlap_ratio >= 0.60:
            ns = min(last["start_idx"], c["start_idx"])
            ne = max(last["end_idx"], c["end_idx"])
            nm = max(ns + 1, ne - contraction_tail + 1)

            b_high = float(np.max(highs[ns:ne + 1]))
            b_low = float(np.min(lows[ns:ne + 1]))
            c_high = float(np.max(highs[nm:ne + 1]))
            c_low = float(np.min(lows[nm:ne + 1]))
            c_ratio = (c_high - c_low) / max(b_high - b_low, 1e-9)

            merged[-1] = {
                **last,
                "start_idx": int(ns),
                "mid_idx": int(nm),
                "end_idx": int(ne),
                "base_high": round(b_high, 2),
                "base_low": round(b_low, 2),
                "contraction_high": round(c_high, 2),
                "contraction_low": round(c_low, 2),
                "contraction_ratio": round(c_ratio, 2),
                "duration": int(ne - ns + 1),
                "retrace_pct": min(last.get("retrace_pct", 1.0), c.get("retrace_pct", 1.0)),
            }
        else:
            merged.append(c.copy())

    return merged


def detect_vcp_from_bases(
    df: pd.DataFrame,
    base_regions: list[dict],
    swings: list[SwingPoint],
    contraction_ratio: float = 0.75,
) -> list[dict]:
    """
    VCP from BASE using swing points with the requested anchor order:
      - Base must exist first.
      - L1 = lowest swing LOW after the base peak.
      - H2 = first swing HIGH registered after L1.
      - L2 = swing LOW after H2.
      - first_retrace  = peak_price - L1.price
      - second_retrace = H2.price - L2.price
      - Valid if second_retrace <= contraction_ratio * first_retrace
        (25% smaller => contraction_ratio = 0.75)
      - Extension rule: extend L2 to later swing LOWs as long as ratio stays valid,
        and keep the latest valid L2.
    """
    out: list[dict] = []

    for b in base_regions:
        p = int(b["peak_idx"])
        s = int(b["start_idx"])
        e = int(b["end_idx"])
        peak_price = float(b["peak_price"])

        if e - s < 4:
            continue

        seq = sorted([sw for sw in swings if s <= sw.index <= e and sw.index > p], key=lambda sw: sw.index)
        if len(seq) < 3:
            continue

        lows_after_peak = [sw for sw in seq if sw.type == "LOW"]
        if not lows_after_peak:
            continue

        # Requested anchor: lowest swing LOW after peak.
        l1 = min(lows_after_peak, key=lambda sw: (sw.price, sw.index))

        highs_after_l1 = [sw for sw in seq if sw.type == "HIGH" and sw.index > l1.index]
        if not highs_after_l1:
            continue

        # Requested anchor: swing HIGH registered after L1 (use first one).
        h2 = highs_after_l1[0]

        first_retrace = peak_price - float(l1.price)
        if first_retrace <= 0:
            continue

        later_lows = [sw for sw in seq if sw.type == "LOW" and sw.index > h2.index]
        if not later_lows:
            continue

        valid_l2 = None
        valid_ratio = None
        valid_second_retrace = None

        # Extension: keep latest later LOW that still satisfies contraction.
        for l2_candidate in later_lows:
            second_retrace_candidate = float(h2.price) - float(l2_candidate.price)
            if second_retrace_candidate <= 0:
                continue

            ratio_candidate = second_retrace_candidate / first_retrace
            if ratio_candidate <= contraction_ratio:
                valid_l2 = l2_candidate
                valid_ratio = ratio_candidate
                valid_second_retrace = second_retrace_candidate

        if valid_l2 is None:
            continue

        l2 = valid_l2
        second_retrace = valid_second_retrace
        ratio = valid_ratio

        out.append({
            "kind": "VCP",
            "peak_idx": int(p),
            "peak_price": round(peak_price, 2),
            "start_idx": int(s),
            "mid_idx": int(h2.index),
            "end_idx": int(l2.index),
            "base_high": float(b["base_high"]),
            "base_low": float(b["base_low"]),
            "contraction_high": round(float(h2.price), 2),
            "contraction_low": round(float(l2.price), 2),
            "l1_idx": int(l1.index),
            "l1_price": round(float(l1.price), 2),
            "h2_idx": int(h2.index),
            "h2_price": round(float(h2.price), 2),
            "l2_idx": int(l2.index),
            "l2_price": round(float(l2.price), 2),
            "first_retrace": round(float(first_retrace), 3),
            "second_retrace": round(float(second_retrace), 3),
            "w2_w1_ratio": round(float(ratio), 4),
            "duration": int(l2.index - s + 1),
            "parent_base_peak_idx": int(p),
        })

    return out


# Source + indicators
src = df_all.copy() if "df_all" in globals() else df.copy()
print(f"Using source ticker: {DATA_TICKER if 'DATA_TICKER' in globals() else TICKER}")
df_cls = _ensure_indicators(_slice_df(src, START_DATE, END_DATE))
if len(df_cls) < 80:
    raise ValueError("Need at least ~80 bars.")

# Swing structure
sig_swings = detect_significant_swings(df_cls, left=3, right=3, prom_atr=1.5, depart_atr=2.5, depart_lookahead=10, min_swing_sep=7)

# Daily base detection + merge
base_raw = detect_daily_base_candidates(df_cls, sig_swings, min_base_duration=20, max_retrace=0.50, contraction_tail=5)
base_merged = merge_overlapping_bases(base_raw, df_cls, contraction_tail=5)


def ensure_base_lows_are_significant_swings(
    df: pd.DataFrame,
    swings: list[SwingPoint],
    base_regions: list[dict],
) -> list[SwingPoint]:
    """If base low is not a significant swing LOW, add a synthetic one at that exact lowest bar."""
    lows = df["low"].values
    atr_vals = df["atr14"].values

    out = list(swings)
    existing_low_idx = {s.index for s in out if s.type == "LOW"}

    for b in base_regions:
        s = int(b["start_idx"])
        e = int(b["end_idx"])
        if e <= s:
            continue

        low_idx = s + int(np.argmin(lows[s : e + 1]))
        low_price = float(lows[low_idx])

        if low_idx in existing_low_idx:
            continue

        atr_ref = float(atr_vals[low_idx]) if np.isfinite(atr_vals[low_idx]) else 0.0
        out.append(SwingPoint(index=low_idx, price=low_price, type="LOW", atr=atr_ref, prominence=0.0))
        existing_low_idx.add(low_idx)

    out.sort(key=lambda sw: sw.index)
    return out


# Enforce: lowest point in each base is a significant swing low.
sig_swings = ensure_base_lows_are_significant_swings(df_cls, sig_swings, base_merged)

# Re-split swings for chart after augmentation
sig_active, sig_exp_high, sig_exp_low = split_swings_for_chart(sig_swings, df_cls)

# VCP from base (second contraction leg must be 25% smaller)
vcp_raw = detect_vcp_from_bases(df_cls, base_merged, sig_swings, contraction_ratio=0.75)
vcp_merged = merge_overlapping_bases(vcp_raw, df_cls, contraction_tail=5)

# Setup signals (safe for fresh kernels)
EMA20_DIST_ATR = globals().get("EMA20_DIST_ATR", 1.0)
EMA20_DEPART_WIN = globals().get("EMA20_DEPART_WIN", 30)
EMA20_TARGET_RR = globals().get("EMA20_TARGET_RR", 3.0)


def _call_detector(name: str, *args, **kwargs):
    fn = globals().get(name)
    if fn is None:
        print(f"[warn] Missing detector: {name} -> using empty list")
        return []
    return fn(*args, **kwargs)


trend_long_strict_all = _call_detector(
    "detect_trend_long_ema20_pullback_strict",
    df_cls,
    sig_swings,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=2.0,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)
trend_long_legacy_all = _call_detector(
    "detect_ema20_pullback",
    df_cls,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=2.0,
    departure_window=EMA20_DEPART_WIN,
    target_rr=EMA20_TARGET_RR,
)
for s in trend_long_legacy_all:
    s.type = "TREND_LONG_20EMA_LEGACY"
    s.metadata["setup_label"] = "Trend Long (20EMA Legacy)"
    s.metadata["direction"] = "LONG"

base_fail_short_all = _call_detector("detect_base_failure_rally_short", df_cls)
trend_short_all = _call_detector(
    "detect_trend_short_ema20_rally",
    df_cls,
    sig_swings,
    dist_atr_max=EMA20_DIST_ATR,
    departure_atr=2.0,
    departure_window=EMA20_DEPART_WIN,
)

last_idx = len(df_cls) - 1

def split_sig(sig):
    return [s for s in sig if s.index == last_idx], [s for s in sig if s.index < last_idx]

trend_long_strict_cur, trend_long_strict_exp = split_sig(trend_long_strict_all)
trend_long_legacy_cur, trend_long_legacy_exp = split_sig(trend_long_legacy_all)
base_fail_cur, base_fail_exp = split_sig(base_fail_short_all)
trend_short_cur, trend_short_exp = split_sig(trend_short_all)

print(f"Base raw: {len(base_raw)} | Base merged: {len(base_merged)}")
print(f"VCP raw: {len(vcp_raw)} | VCP merged: {len(vcp_merged)}")
print(f"Expired significant swings: high={len(sig_exp_high)} low={len(sig_exp_low)}")
print(f"Expired setups: trend20_strict={len(trend_long_strict_exp)}, trend20_legacy={len(trend_long_legacy_exp)}, base_fail={len(base_fail_exp)}, trend_short={len(trend_short_exp)}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.76, 0.24], vertical_spacing=0.03)
fig.add_trace(go.Candlestick(x=df_cls["date"], open=df_cls["open"], high=df_cls["high"], low=df_cls["low"], close=df_cls["close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["ema20"], name="EMA 20", line=dict(color="#f59e0b", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["sma50"], name="SMA 50", line=dict(color="#3b82f6", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_cls["date"], y=df_cls["sma200"], name="SMA 200", line=dict(color="#ef4444", width=1.5)), row=1, col=1)

# VCP is shown using rectangles only (no per-bar VCP markers)

# swings active
for arr, typ, name, symbol, color, size in [
    ([s for s in sig_active if s.type=="HIGH"], "H", "Sig Swing High", "star", "#dc2626", 10),
    ([s for s in sig_active if s.type=="LOW"], "L", "Sig Swing Low", "star", "#16a34a", 10),
]:
    if arr:
        fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in arr], y=[s.price for s in arr], mode="markers", name=name, marker=dict(symbol=symbol, size=size, color=color, line=dict(width=1, color="white") if symbol=="star" else None)), row=1, col=1)

# swings expired (significant only)
if sig_exp_high:
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in sig_exp_high], y=[s.price for s in sig_exp_high], mode="markers", name="Expired Swing High", marker=dict(symbol="x", size=8, color="rgba(239,68,68,0.60)")), row=1, col=1)
if sig_exp_low:
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in sig_exp_low], y=[s.price for s in sig_exp_low], mode="markers", name="Expired Swing Low", marker=dict(symbol="x", size=8, color="rgba(34,197,94,0.60)")), row=1, col=1)

# merged base rectangles
DRAW_BASE_RECTANGLES = True
BASE_RECT_MAX = 20
if DRAW_BASE_RECTANGLES:
    for b in base_merged[-BASE_RECT_MAX:]:
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[b["start_idx"]], x1=df_cls["date"].iloc[b["end_idx"]], y0=b["base_low"], y1=b["base_high"], xref="x", yref="y", line=dict(color="rgba(34,197,94,0.45)", width=1), fillcolor="rgba(34,197,94,0.08)", layer="below")
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[b["mid_idx"]], x1=df_cls["date"].iloc[b["end_idx"]], y0=b["contraction_low"], y1=b["contraction_high"], xref="x", yref="y", line=dict(color="rgba(16,185,129,0.75)", width=1), fillcolor="rgba(16,185,129,0.16)", layer="below")
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[-1]], y=[df_cls["close"].iloc[-1]], mode="markers", name="Daily Base (merged) + Contraction", marker=dict(symbol="square", size=10, color="rgba(16,185,129,0.35)", line=dict(width=1, color="rgba(16,185,129,0.9)")), visible="legendonly"), row=1, col=1)

# merged VCP rectangles (blue)
DRAW_VCP_RECTANGLES = True
VCP_RECT_MAX = 20
if DRAW_VCP_RECTANGLES:
    for v in vcp_merged[-VCP_RECT_MAX:]:
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[v["start_idx"]], x1=df_cls["date"].iloc[v["end_idx"]], y0=v["base_low"], y1=v["base_high"], xref="x", yref="y", line=dict(color="rgba(59,130,246,0.55)", width=1), fillcolor="rgba(59,130,246,0.08)", layer="below")
        fig.add_shape(type="rect", x0=df_cls["date"].iloc[v["h2_idx"]], x1=df_cls["date"].iloc[v["l2_idx"]], y0=v["l2_price"], y1=v["h2_price"], xref="x", yref="y", line=dict(color="rgba(37,99,235,0.9)", width=1), fillcolor="rgba(37,99,235,0.18)", layer="below")
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[-1]], y=[df_cls["close"].iloc[-1]], mode="markers", name="VCP (W2 <= 0.75*W1)", marker=dict(symbol="square", size=10, color="rgba(37,99,235,0.35)", line=dict(width=1, color="rgba(37,99,235,0.95)")), visible="legendonly"), row=1, col=1)

STYLE = {
    "TREND_LONG_20EMA_PULLBACK": {"cur": ("diamond", "#10b981", "Trend Long Strict (Current)"), "exp": ("x", "rgba(16,185,129,0.5)", "Trend Long Strict (Expired)")},
    "TREND_LONG_20EMA_LEGACY": {"cur": ("diamond", "#84cc16", "Trend Long Legacy (Current)"), "exp": ("x", "rgba(132,204,22,0.5)", "Trend Long Legacy (Expired)")},
    "BASE_FAILURE_SHORT": {"cur": ("diamond", "#ef4444", "Base Failure Short (Current)"), "exp": ("x", "rgba(239,68,68,0.5)", "Base Failure Short (Expired)")},
    "TREND_SHORT_20EMA_RALLY": {"cur": ("diamond", "#f97316", "Trend Short (Current)"), "exp": ("x", "rgba(249,115,22,0.5)", "Trend Short (Expired)")},
}


def add_sig(sig_list, key, state):
    if not sig_list:
        return
    symbol, color, name = STYLE[key][state]
    fig.add_trace(go.Scatter(x=[df_cls["date"].iloc[s.index] for s in sig_list], y=[s.price for s in sig_list], mode="markers", name=name, marker=dict(symbol=symbol, size=12 if state=="cur" else 9, color=color, line=dict(width=2 if state=="cur" else 1, color="white"))), row=1, col=1)

add_sig(trend_long_strict_exp, "TREND_LONG_20EMA_PULLBACK", "exp")
add_sig(trend_long_legacy_exp, "TREND_LONG_20EMA_LEGACY", "exp")
add_sig(base_fail_exp, "BASE_FAILURE_SHORT", "exp")
add_sig(trend_short_exp, "TREND_SHORT_20EMA_RALLY", "exp")

add_sig(trend_long_strict_cur, "TREND_LONG_20EMA_PULLBACK", "cur")
add_sig(trend_long_legacy_cur, "TREND_LONG_20EMA_LEGACY", "cur")
add_sig(base_fail_cur, "BASE_FAILURE_SHORT", "cur")
add_sig(trend_short_cur, "TREND_SHORT_20EMA_RALLY", "cur")

vol_colors = ["#22c55e" if df_cls["close"].iloc[i] >= df_cls["open"].iloc[i] else "#ef4444" for i in range(len(df_cls))]
fig.add_trace(go.Bar(x=df_cls["date"], y=df_cls["volume"], name="Volume", marker_color=vol_colors, opacity=0.45), row=2, col=1)

fig.update_layout(title=f"{TICKER} — Daily Base + All Expired Overlays", template="plotly_dark", height=860, xaxis_rangeslider_visible=False, legend=dict(orientation="h", y=1.02, x=0))
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show(renderer="notebook_connected")

Using source ticker: GDX
Base raw: 4 | Base merged: 2
VCP raw: 2 | VCP merged: 2
Expired significant swings: high=19 low=9
Expired setups: trend20_strict=11, trend20_legacy=30, base_fail=36, trend_short=1
